# EvolBA — Evolutionary Boundary Attack on Fashion-MNIST

**Setting:** Hard-Label Black-Box (HL-BB) — the attacker receives only the predicted label. No gradients, no probabilities, no model internals.

**Method:** EvolBA is a *decision-based* attack. It operates in three phases:

| Phase | What happens |
|---|---|
| **1 — Init** | Find any adversarial example via random perturbations |
| **2 — Boundary** | Binary search between clean and adversarial to land on the decision boundary |
| **3 — Refine** | CMA-ES initialized at the boundary point minimises `‖x_adv − x‖` while staying adversarial |

The key difference from a naive CMA-ES attack: EvolBA always **starts on the adversarial side** of the decision boundary and walks **toward** the original image, so feasibility is guaranteed from the first iteration.

## 1. Setup

In [1]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib.colors import TwoSlopeNorm

import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
import numpy as np
import cma
from pathlib import Path

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
SEED   = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

CLASS_NAMES = [
    'T-shirt', 'Trouser', 'Pullover', 'Dress', 'Coat',
    'Sandal',  'Shirt',   'Sneaker',  'Bag',   'Ankle boot'
]

print(f'Device : {DEVICE}')
print(f'CMA-ES : {cma.__version__}')

Device : cpu
CMA-ES : 4.4.4


## 2. Dataset

In [2]:
transform = transforms.ToTensor()

train_ds = torchvision.datasets.FashionMNIST(root='./data', train=True,  download=True, transform=transform)
test_ds  = torchvision.datasets.FashionMNIST(root='./data', train=False, download=True, transform=transform)

train_loader = torch.utils.data.DataLoader(train_ds, batch_size=128, shuffle=True,  num_workers=0)
test_loader  = torch.utils.data.DataLoader(test_ds,  batch_size=128, shuffle=False, num_workers=0)

print(f'Train: {len(train_ds):,}  |  Test: {len(test_ds):,}')

Train: 60,000  |  Test: 10,000


## 3. Classifier

In [3]:
class SmallCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 32, 3, padding=1), nn.ReLU(),
            nn.Conv2d(32, 32, 3, padding=1), nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(),
            nn.Conv2d(64, 64, 3, padding=1), nn.ReLU(),
            nn.MaxPool2d(2),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * 7 * 7, 256), nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, 10),
        )

    def forward(self, x):
        return self.classifier(self.features(x))


WEIGHTS = Path('cnn_fashionmnist.pth')
model   = SmallCNN().to(DEVICE)

if WEIGHTS.exists():
    model.load_state_dict(torch.load(WEIGHTS, map_location=DEVICE))
    print('Loaded saved weights — skipping training.')
else:
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=1e-3)
    scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.5)
    for epoch in range(1, 11):
        model.train()
        loss_sum, correct = 0.0, 0
        for imgs, labels in train_loader:
            imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
            optimizer.zero_grad()
            out  = model(imgs)
            loss = criterion(out, labels)
            loss.backward()
            optimizer.step()
            loss_sum += loss.item() * len(imgs)
            correct  += (out.argmax(1) == labels).sum().item()
        scheduler.step()
        print(f'Epoch {epoch:2d}/10  loss={loss_sum/len(train_ds):.4f}  acc={correct/len(train_ds)*100:.2f}%')
    torch.save(model.state_dict(), WEIGHTS)
    print(f'Weights saved to {WEIGHTS}')

model.eval()
correct, total = 0, 0
with torch.no_grad():
    for imgs, labels in test_loader:
        preds    = model(imgs.to(DEVICE)).argmax(1).cpu()
        correct += (preds == labels).sum().item()
        total   += len(labels)
print(f'Test accuracy: {correct/total*100:.2f}%')

Epoch  1/10  loss=0.5452  acc=79.98%
Epoch  2/10  loss=0.3203  acc=88.17%
Epoch  3/10  loss=0.2727  acc=89.97%
Epoch  4/10  loss=0.2378  acc=91.22%
Epoch  5/10  loss=0.2128  acc=92.15%
Epoch  6/10  loss=0.1766  acc=93.51%
Epoch  7/10  loss=0.1611  acc=94.05%
Epoch  8/10  loss=0.1465  acc=94.48%
Epoch  9/10  loss=0.1367  acc=94.83%
Epoch 10/10  loss=0.1250  acc=95.33%
Weights saved to cnn_fashionmnist.pth
Test accuracy: 92.79%


## 4. Collect correctly-classified test samples

Attacks are only meaningful on samples the model already gets right.

In [4]:
def collect_correct(model, dataset, n=200):
    model.eval()
    samples = []
    loader  = torch.utils.data.DataLoader(dataset, batch_size=64, shuffle=False)
    with torch.no_grad():
        for imgs, labels in loader:
            preds   = model(imgs.to(DEVICE)).argmax(1).cpu()
            correct = (preds == labels).nonzero(as_tuple=True)[0]
            for i in correct:
                samples.append((imgs[i], labels[i].item()))
                if len(samples) >= n:
                    return samples
    return samples

SAMPLES = collect_correct(model, test_ds, n=200)
print(f'{len(SAMPLES)} correctly-classified samples ready.')

200 correctly-classified samples ready.


## 5. EvolBA Implementation

### 5.1 HL-BB oracle

The only channel of information from the model to the attacker.

In [5]:
def oracle(model, image_np):
    """Hard-label black-box query: returns predicted class index only."""
    t = torch.from_numpy(image_np).float().unsqueeze(0).unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        return model(t).argmax(1).item()

### 5.2 Phase 1 — Find an initial adversarial example

EvolBA needs a starting point that is already **misclassified**. The exact perturbation size does not matter here — we just need *any* adversarial image. We obtain one by adding Gaussian noise with increasing scale until the model is fooled.

In [6]:
def find_initial_adversarial(model, image_np, true_label, query_counter, max_attempts=500):
    """
    Add random Gaussian noise with growing scale until the model misclassifies.

    Returns the first adversarial image found, or None if budget is exhausted.
    query_counter is a mutable [int] shared with the caller.
    """
    rng = np.random.default_rng(SEED)
    for scale in np.linspace(0.05, 2.0, max_attempts):
        noise = rng.standard_normal(image_np.shape) * scale
        adv   = np.clip(image_np + noise, 0.0, 1.0)
        query_counter[0] += 1
        if oracle(model, adv) != true_label:
            return adv
    return None

### 5.3 Phase 2 — Binary search to the decision boundary

We have a clean image `x` (correctly classified) and an adversarial `x_adv` (misclassified). Binary search on the segment between them finds the point **as close as possible to the boundary on the adversarial side**.

This is the initialisation for CMA-ES: starting from the boundary rather than from a distant adversarial point gives the optimiser the best possible head start.

```
   x ──────────────────────|──── x_adv
  (clean)             boundary   (adversarial, far)
                             ↑
                       binary search result
```

In [7]:
def binary_search_boundary(model, image_np, adv_np, true_label, query_counter, n_steps=15):
    """
    Binary search on [image_np, adv_np] for the decision boundary.

    Invariant:
        lo  →  correctly classified  (clean side)
        hi  →  misclassified         (adversarial side)

    Returns the hi endpoint at convergence: the closest adversarial to x.
    """
    lo = image_np.copy()
    hi = adv_np.copy()

    for _ in range(n_steps):
        mid = (lo + hi) / 2.0
        query_counter[0] += 1
        if oracle(model, mid) != true_label:
            hi = mid   # still adversarial → push toward clean image
        else:
            lo = mid   # became clean → push back

    return hi

### 5.4 Phase 3 — CMA-ES boundary refinement

CMA-ES is initialised at `δ_boundary = x_boundary − x`. Its objective is to **minimise `‖δ‖₂`** while keeping `f(x + δ) ≠ y`.

**Fitness function:**
$$
F(\delta) = \begin{cases}
\|x_{\text{clip}} - x\|_2 & \text{if } f(x_{\text{clip}}) \neq y \quad\text{(adversarial — minimise distance)}\\
C + \|x_{\text{clip}} - x\|_2 & \text{otherwise} \quad\text{(clean — large penalty)}
\end{cases}
$$
where $x_{\text{clip}} = \text{clip}(x + \delta,\, 0, 1)$.

The constant `C` is larger than any realistic L2 norm, so the ranking is always:
*any adversarial candidate* ≺ *any clean candidate*.

**Initial step size** `σ₀` is set proportional to the boundary perturbation norm so the initial population meaningfully spans the region around the boundary point.

In [8]:
PENALTY = 20.0   # must exceed any realistic ||x_adv - x||_2 for 28×28 images in [0,1]

def cmaes_refine(model, image_np, true_label, boundary_adv,
                 query_counter, max_queries, gen_l2_history):
    """
    Phase 3: CMA-ES refinement starting from the boundary adversarial.

    Args:
        boundary_adv     : (28,28) adversarial image from binary search
        query_counter    : mutable [int] shared query count
        max_queries      : total budget across all phases
        gen_l2_history   : list; best L2 per generation is appended here

    Returns:
        best_adv (ndarray), best_l2 (float)
    """
    delta_init = (boundary_adv - image_np).flatten()   # shape (784,)
    n          = delta_init.size
    init_l2    = float(np.linalg.norm(delta_init))

    best_adv = boundary_adv.copy()
    best_l2  = init_l2

    # σ₀: fraction of the initial perturbation norm — covers the neighbourhood
    # around the boundary point without overshooting back into the clean region.
    sigma0 = max(0.3 * init_l2 / np.sqrt(n), 1e-3)

    remaining = max_queries - query_counter[0]
    if remaining <= 0:
        return best_adv, best_l2

    es = cma.CMAEvolutionStrategy(
        delta_init,
        sigma0,
        {'maxfevals': remaining, 'verbose': -9, 'seed': SEED}
    )

    while not es.stop() and query_counter[0] < max_queries:
        candidates = es.ask()
        fitnesses  = []

        for c in candidates:
            x_cand = np.clip(image_np + c.reshape(image_np.shape), 0.0, 1.0)
            query_counter[0] += 1
            pred = oracle(model, x_cand)
            l2   = float(np.linalg.norm(x_cand - image_np))

            if pred != true_label:
                if l2 < best_l2:
                    best_l2  = l2
                    best_adv = x_cand.copy()
                fitnesses.append(l2)
            else:
                fitnesses.append(PENALTY + l2)

        es.tell(candidates, fitnesses)
        gen_l2_history.append(best_l2)

    return best_adv, best_l2

### 5.5 Full EvolBA pipeline

In [9]:
def evolba(model, image_np, true_label, max_queries=2000, bs_steps=15):
    """
    EvolBA: Evolutionary Boundary Attack (hard-label black-box).

    Args:
        image_np    : (28, 28) float32 in [0, 1], correctly classified by model
        true_label  : ground-truth class index
        max_queries : total oracle query budget across all three phases
        bs_steps    : number of binary-search steps in Phase 2

    Returns dict:
        success         : bool
        queries         : int — total queries used
        best_adv        : (28,28) final adversarial image
        best_l2         : float — L2 of final perturbation
        init_l2         : float — L2 after binary search (Phase 2 output)
        l2_reduction    : float — percentage improvement (init → final)
        gen_l2_history  : list[float] — best L2 per CMA-ES generation
    """
    qc = [0]   # mutable query counter shared across all phases

    # ── Phase 1: find any adversarial example ─────────────────────────────
    init_adv = find_initial_adversarial(model, image_np, true_label, qc)
    if init_adv is None:
        return {'success': False, 'queries': qc[0]}

    # ── Phase 2: binary search to the decision boundary ───────────────────
    boundary_adv = binary_search_boundary(
        model, image_np, init_adv, true_label, qc, n_steps=bs_steps
    )
    init_l2 = float(np.linalg.norm(boundary_adv - image_np))

    # ── Phase 3: CMA-ES refinement ────────────────────────────────────────
    gen_l2_history = []
    best_adv, best_l2 = cmaes_refine(
        model, image_np, true_label, boundary_adv, qc, max_queries, gen_l2_history
    )

    reduction = (init_l2 - best_l2) / init_l2 * 100 if init_l2 > 0 else 0.0

    return {
        'success'        : True,
        'queries'        : qc[0],
        'best_adv'       : best_adv,
        'best_l2'        : best_l2,
        'init_l2'        : init_l2,
        'l2_reduction'   : reduction,
        'gen_l2_history' : gen_l2_history,
    }

## 6. Single-image demo

Run EvolBA on one image and inspect how the perturbation shrinks across the three phases.

In [10]:
MAX_Q    = 2000
BS_STEPS = 15

demo_img, demo_label = SAMPLES[0]
demo_np = demo_img.squeeze().numpy()

print(f'True class : {CLASS_NAMES[demo_label]}')
r_demo = evolba(model, demo_np, demo_label, max_queries=MAX_Q, bs_steps=BS_STEPS)

print(f"Queries used                   : {r_demo['queries']}")
print(f"L2 after binary search (Phase 2): {r_demo['init_l2']:.4f}")
print(f"L2 after CMA-ES (Phase 3)       : {r_demo['best_l2']:.4f}")
print(f"L2 reduction                    : {r_demo['l2_reduction']:.1f}%")
if r_demo['success']:
    adv_label = oracle(model, r_demo['best_adv'])
    print(f"Model now predicts              : {CLASS_NAMES[adv_label]}")

True class : Ankle boot
Queries used                   : 2018
L2 after binary search (Phase 2): 3.3216
L2 after CMA-ES (Phase 3)       : 1.8517
L2 reduction                    : 44.3%
Model now predicts              : Bag


In [11]:
# ── Convergence: best L2 per CMA-ES generation ───────────────────────────
gen_hist = r_demo['gen_l2_history']

fig, ax = plt.subplots(figsize=(8, 3.5))
ax.plot(gen_hist, color='#2c7bb6', linewidth=1.5, label='Best L2 (CMA-ES phase)')
ax.axhline(r_demo['init_l2'], color='#fdae61', linewidth=1.2, linestyle='--',
           label=f"Phase 2 boundary  L2={r_demo['init_l2']:.3f}")
ax.axhline(r_demo['best_l2'], color='#1a9641', linewidth=1.2, linestyle=':',
           label=f"Phase 3 final     L2={r_demo['best_l2']:.3f}")
ax.set_xlabel('CMA-ES generation')
ax.set_ylabel('Best L2 norm')
ax.set_title(f'EvolBA convergence — "{CLASS_NAMES[demo_label]}"  (budget={MAX_Q})')
ax.legend(fontsize=8)
ax.grid(True, alpha=0.25)
plt.tight_layout()
plt.savefig('convergence_demo.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved → convergence_demo.png')

Saved → convergence_demo.png


C:\Users\tomma\AppData\Local\Temp\ipykernel_21860\3768934643.py:17: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### 6.1 Visual inspection — original vs adversarial vs perturbation

In [12]:
adv_np    = r_demo['best_adv']
delta_np  = adv_np - demo_np
adv_label = oracle(model, adv_np)
vmax      = max(abs(delta_np).max(), 1e-6)

fig, axes = plt.subplots(1, 3, figsize=(9, 3.2))
fig.patch.set_facecolor('#1a1a2e')

axes[0].imshow(demo_np, cmap='gray', vmin=0, vmax=1)
axes[0].set_title(f'Original\n{CLASS_NAMES[demo_label]}',
                  color='white', fontsize=10, fontweight='bold',
                  bbox=dict(facecolor='#0d6efd', alpha=0.85, pad=3, edgecolor='none'))
axes[0].axis('off')

axes[1].imshow(adv_np, cmap='gray', vmin=0, vmax=1)
axes[1].set_title(f'Adversarial\n→ {CLASS_NAMES[adv_label]}',
                  color='white', fontsize=10, fontweight='bold',
                  bbox=dict(facecolor='#dc3545', alpha=0.85, pad=3, edgecolor='none'))
axes[1].axis('off')

norm = TwoSlopeNorm(vmin=-vmax, vcenter=0, vmax=vmax)
im   = axes[2].imshow(delta_np, cmap='bwr', norm=norm)
axes[2].set_title(f'Perturbation \u0394\nL2={r_demo["best_l2"]:.3f}',
                  color='#cccccc', fontsize=10)
axes[2].axis('off')
fig.colorbar(im, ax=axes[2], fraction=0.046, pad=0.04)

plt.suptitle('EvolBA — single image demo', color='white', fontsize=11, y=1.02)
plt.tight_layout()
plt.savefig('demo_single.png', dpi=150, bbox_inches='tight', facecolor=fig.get_facecolor())
plt.show()
print('Saved → demo_single.png')

Saved → demo_single.png


C:\Users\tomma\AppData\Local\Temp\ipykernel_21860\1128827093.py:31: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 7. Batch evaluation

In [13]:
N_EVAL  = 50   # increase for more reliable statistics
results = []

for i, (img, label) in enumerate(SAMPLES[:N_EVAL]):
    img_np = img.squeeze().numpy()
    r = evolba(model, img_np, label, max_queries=MAX_Q, bs_steps=BS_STEPS)
    r['true_label'] = label
    r['image_np']   = img_np
    if r['success']:
        r['adv_label'] = oracle(model, r['best_adv'])
    results.append(r)
    if (i + 1) % 10 == 0:
        succ    = [x for x in results if x['success']]
        avg_l2  = np.mean([x['best_l2'] for x in succ]) if succ else float('nan')
        print(f'[{i+1:3d}/{N_EVAL}]  success={len(succ)/len(results)*100:.0f}%  avg_L2={avg_l2:.4f}')

successes = [r for r in results if r['success']]
print(f'\n── Final summary ───────────────────────────────────')
print(f'Success rate       : {len(successes)/N_EVAL*100:.1f}%  ({len(successes)}/{N_EVAL})')
print(f'Avg queries        : {np.mean([r["queries"]       for r in successes]):.0f}')
print(f'Avg L2 (boundary)  : {np.mean([r["init_l2"]      for r in successes]):.4f}')
print(f'Avg L2 (final)     : {np.mean([r["best_l2"]      for r in successes]):.4f}')
print(f'Avg L2 reduction   : {np.mean([r["l2_reduction"] for r in successes]):.1f}%')

[ 10/50]  success=100%  avg_L2=2.1574
[ 20/50]  success=100%  avg_L2=1.8231
[ 30/50]  success=100%  avg_L2=1.7681
[ 40/50]  success=100%  avg_L2=1.7263
[ 50/50]  success=100%  avg_L2=1.6838

── Final summary ───────────────────────────────────
Success rate       : 100.0%  (50/50)
Avg queries        : 2012
Avg L2 (boundary)  : 2.8847
Avg L2 (final)     : 1.6838
Avg L2 reduction   : 41.6%


## 8. Visualisations

### 8.1 Adversarial gallery

**Row 0** — original image, true label (blue badge)  
**Row 1** — adversarial image, model prediction (red badge) — visually identical to the human eye  
**Row 2** — perturbation heatmap: blue = pixels pushed down, red = pixels pushed up

In [14]:
N_SHOW = min(8, len(successes))
show   = successes[:N_SHOW]

fig, axes = plt.subplots(3, N_SHOW, figsize=(2.2 * N_SHOW, 7.5))
fig.patch.set_facecolor('#1a1a2e')

for col, r in enumerate(show):
    orig  = r['image_np']
    adv   = r['best_adv']
    delta = adv - orig
    tl    = CLASS_NAMES[r['true_label']]
    al    = CLASS_NAMES[r['adv_label']]
    vmax  = max(abs(delta).max(), 1e-6)

    axes[0, col].imshow(orig, cmap='gray', vmin=0, vmax=1, interpolation='nearest')
    axes[0, col].set_title(tl, fontsize=8, color='white', fontweight='bold',
                            bbox=dict(facecolor='#0d6efd', alpha=0.85, pad=2, edgecolor='none'))
    axes[0, col].axis('off')

    axes[1, col].imshow(adv, cmap='gray', vmin=0, vmax=1, interpolation='nearest')
    axes[1, col].set_title(f'\u2192 {al}', fontsize=8, color='white', fontweight='bold',
                            bbox=dict(facecolor='#dc3545', alpha=0.85, pad=2, edgecolor='none'))
    axes[1, col].axis('off')

    norm = TwoSlopeNorm(vmin=-vmax, vcenter=0, vmax=vmax)
    axes[2, col].imshow(delta, cmap='bwr', norm=norm, interpolation='nearest')
    axes[2, col].set_title(
        f'L2={r["best_l2"]:.2f}  ↓{r["l2_reduction"]:.0f}%\nq={r["queries"]}',
        fontsize=7, color='#bbbbbb'
    )
    axes[2, col].axis('off')

for row_i, lbl in enumerate(['Original', 'Adversarial', 'Perturbation \u0394']):
    axes[row_i, 0].set_ylabel(lbl, fontsize=9, color='white', rotation=90, labelpad=6)

fig.suptitle(f'EvolBA Adversarial Gallery  |  budget={MAX_Q}  n={N_SHOW}',
             fontsize=11, color='white', y=1.01)
plt.tight_layout()
plt.savefig('adversarial_gallery.png', dpi=150, bbox_inches='tight',
            facecolor=fig.get_facecolor())
plt.show()
print('Saved \u2192 adversarial_gallery.png')

Saved → adversarial_gallery.png


C:\Users\tomma\AppData\Local\Temp\ipykernel_21860\3097162479.py:41: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### 8.2 L2 reduction: boundary → CMA-ES

Each dot is one attacked image. Points **below the diagonal** mean CMA-ES successfully shrank the perturbation; color encodes total queries used.

In [15]:
init_l2s  = [r['init_l2']      for r in successes]
final_l2s = [r['best_l2']      for r in successes]
queries_s = [r['queries']      for r in successes]
reduct_s  = [r['l2_reduction'] for r in successes]

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

# Scatter: init L2 vs final L2
sc  = axes[0].scatter(init_l2s, final_l2s, c=queries_s, cmap='plasma',
                      s=55, alpha=0.85, edgecolors='white', linewidths=0.4)
lim = max(max(init_l2s), max(final_l2s)) * 1.06
axes[0].plot([0, lim], [0, lim], 'k--', linewidth=0.9, alpha=0.5, label='No improvement')
axes[0].set_xlabel('L2 after binary search (Phase 2)')
axes[0].set_ylabel('L2 after CMA-ES (Phase 3)')
axes[0].set_title('Perturbation reduction by CMA-ES')
axes[0].legend(fontsize=8)
axes[0].grid(True, alpha=0.25)
fig.colorbar(sc, ax=axes[0], label='Total queries')

# Bar: mean L2 at each stage
stages = ['Binary search\n(Phase 2)', 'CMA-ES\n(Phase 3)']
means  = [np.mean(init_l2s), np.mean(final_l2s)]
stds   = [np.std(init_l2s),  np.std(final_l2s)]
bars   = axes[1].bar(stages, means, yerr=stds,
                     color=['#fdae61', '#1a9641'],
                     alpha=0.85, capsize=5, edgecolor='white')
axes[1].bar_label(bars, fmt='%.3f', fontsize=9, padding=4)
axes[1].set_ylabel('Mean L2 norm')
axes[1].set_title(f'Mean L2 reduction: {np.mean(reduct_s):.1f}%')
axes[1].set_ylim(0, max(means) * 1.3)
axes[1].grid(axis='y', alpha=0.25)

fig.suptitle(f'EvolBA — perturbation analysis  (n={len(successes)} attacks)', fontsize=11)
plt.tight_layout()
plt.savefig('l2_reduction.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved \u2192 l2_reduction.png')

Saved → l2_reduction.png


C:\Users\tomma\AppData\Local\Temp\ipykernel_21860\375614199.py:36: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### 8.3 Query distribution and success rate vs budget

In [16]:
budgets = [100, 250, 500, 750, 1000, 1500, MAX_Q]
sr_at   = [
    sum(1 for r in results if r['success'] and r['queries'] <= b) / N_EVAL * 100
    for b in budgets
]

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

axes[0].hist(queries_s, bins=15, color='#2c7bb6', edgecolor='white', linewidth=0.5)
axes[0].axvline(np.mean(queries_s), color='#d7191c', linestyle='--', linewidth=1.5,
                label=f'Mean = {np.mean(queries_s):.0f}')
axes[0].set_xlabel('Queries used')
axes[0].set_ylabel('Count')
axes[0].set_title('Distribution of queries per attack')
axes[0].legend()
axes[0].grid(True, alpha=0.25)

axes[1].plot(budgets, sr_at, 'o-', color='#2c7bb6', linewidth=2, markersize=6)
axes[1].fill_between(budgets, sr_at, alpha=0.15, color='#2c7bb6')
for b, sr in zip(budgets, sr_at):
    axes[1].annotate(f'{sr:.0f}%', (b, sr), textcoords='offset points',
                     xytext=(0, 7), ha='center', fontsize=8)
axes[1].set_xlabel('Query budget')
axes[1].set_ylabel('Success rate (%)')
axes[1].set_title('Success rate vs query budget')
axes[1].set_ylim(0, 108)
axes[1].grid(True, alpha=0.25)

fig.suptitle(f'EvolBA — query efficiency  (n={N_EVAL})', fontsize=11)
plt.tight_layout()
plt.savefig('query_stats.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved \u2192 query_stats.png')

Saved → query_stats.png


C:\Users\tomma\AppData\Local\Temp\ipykernel_21860\4058862371.py:32: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
